# CarbonGlobe — Weighted ensemble of the three trained models

**IEEE Big Data Cup 2026 — CarbonGlobe AI Emulation Challenge**

The three runs already beat the baseline (LB 0.231):

| run | file (`*_preds.npz`) | LB |
|---|---|---|
| MM-LSTM-Age v2 + scheduled sampling | `v2_scheduled_sampling_preds.npz` | 0.186 |
| EcoFormer (transformer ED) | `ecoformer_ed_preds.npz` | 0.169 |
| LSTM-XL (deeper DeepED) | `lstm_xl_preds.npz` | 0.168 |

They use **different inductive biases**, so their errors are partly independent — averaging their
year-40 predictions usually beats any single model. This notebook:

1. loads the saved `val_pred` + test predictions from each run,
2. tunes non-negative blend weights on the validation set with the real metric,
3. also tries a per-variable (height / AGB separate) blend,
4. writes `submission.csv` from the best blend.

No training here — it only needs the `*_preds.npz` files. Attach each run's Kaggle output (or the
dataset you saved them to) so the paths below resolve. If a file is missing, the notebook re-creates
it from that run's saved `*.weights.h5` (needs the model definition; see the note in section 3).

## 0. Setup & competition metric

In [ ]:
import os, glob, itertools, gc
import numpy as np
import pandas as pd

OUT = '/kaggle/working'
print('numpy', np.__version__)

## 1. Locate and load each run's predictions

Each `*_preds.npz` holds `val_pred` (15, N_val, 7) and one array per scenario (`ssp126`, `ssp585`),
all in **physical units**, year-40. We search a few common Kaggle input locations for each file.

In [ ]:
# run_name -> the LB score you observed (used only to seed the search & sanity-print).
# Any run whose *_preds.npz is missing is skipped automatically, so you can list all of them.
RUNS = {
    'v2_scheduled_sampling': 0.186,
    'ecoformer_ed':          0.169,
    'lstm_xl':               0.168,
    'fusion_v2':             0.149,   # current best
    # deeper models (add once you have run them):
    'crossvar_transformer':  None,
    'lstm_xxl':              None,
    'tft':                   None,
}

SEARCH_DIRS = ['/kaggle/working', '/kaggle/input'] + glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')

def find_preds(run):
    fname = f'{run}_preds.npz'
    for d in SEARCH_DIRS:
        p = os.path.join(d, fname)
        if os.path.isfile(p):
            return p
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    return hits[0] if hits else None

preds = {}
for run in RUNS:
    p = find_preds(run)
    if p is None:
        print(f'  !! {run}: {run}_preds.npz NOT found — attach its output/dataset (see section 3)')
        continue
    d = np.load(p)
    preds[run] = {k: d[k].astype(np.float32) for k in d.files}
    print(f'  ok {run}: {p}  keys={list(d.files)}  val_pred={preds[run]["val_pred"].shape}')

assert preds, 'No prediction files found. Attach the runs\' outputs as Kaggle datasets.'
present = list(preds.keys())
print('\nusable runs:', present)

## 2. Validation ground truth (year-40)

In [ ]:
# Rebuild the same held-out validation year-40 truth the runs scored against.
BASE = '/kaggle/input/datasets/zhihaow/carbonglobe'
stats = np.load(f'{BASE}/data_stats/data_stats.npz')
Y_MEAN = stats['y_mean'].astype(np.float32)

res = np.load(f'{BASE}/data_global/res_train4_test8.npz', mmap_mode='r')
val_true = np.ascontiguousarray(res['y_test'][..., 11, :])[:, :, 40, :].astype(np.float32)  # (15,Nval,7)
res.close(); gc.collect()
print('val_true', val_true.shape)

def scored_metric(pred, true):
    ph, th = pred[..., 0] / Y_MEAN[0], true[..., 0] / Y_MEAN[0]
    pa, ta = pred[..., 1] / Y_MEAN[1], true[..., 1] / Y_MEAN[1]
    mse_h = float(np.mean((ph - th) ** 2)); mse_a = float(np.mean((pa - ta) ** 2))
    return 0.5 * (mse_h + mse_a), mse_h, mse_a

print('\nsingle-model VAL metric (should track your LB ordering):')
for run in present:
    m, mh, ma = scored_metric(preds[run]['val_pred'], val_true)
    print(f'  {run:26s} VAL {m:.5f}  (h {mh:.4f}, agb {ma:.4f})')

## 3. (Optional) recreate a missing `*_preds.npz`

If a run's prediction file is unavailable but you kept its `*.weights.h5`, re-run that run's notebook
(sections defining the model + `rollout`) and re-save the `.npz`. That is cheaper than retraining:
only the 40-year rollout is needed, not the fit. The blend below simply skips any missing run, so you
can still ensemble the two you have.

## 4. Tune blend weights on the validation set

In [ ]:
val_stack = np.stack([preds[r]['val_pred'] for r in present], axis=0)   # (R,15,Nval,7)
R = len(present)

def blend_metric(weights, per_var=None):
    w = np.asarray(weights, np.float32)
    if per_var is None:
        p = np.tensordot(w, val_stack, axes=(0, 0))
    else:  # per_var: (R,2) weights for [height, agb]; other vars use the mean
        p = val_stack.mean(0).copy()
        for j in range(2):
            p[..., j] = np.tensordot(per_var[:, j], val_stack[..., j], axes=(0, 0))
    return scored_metric(p, val_true)[0]

# --- baseline references ---
best = {'name': 'equal mean', 'w': np.ones(R) / R}
best['score'] = blend_metric(best['w'])
print(f'equal mean        VAL {best["score"]:.5f}   w={np.round(best["w"],3)}')

# --- coarse simplex grid search over non-negative weights that sum to 1 ---
grid = np.linspace(0, 1, 21)
def simplex(n, g):
    if n == 1:
        yield (1.0,)
        return
    for a in g:
        for rest in simplex(n - 1, g):
            s = sum(rest)
            if s <= 1e-9:
                if abs(a - 1.0) < 1e-9:
                    yield (a,) + rest
            else:
                yield (a,) + tuple(r * (1 - a) for r in rest)

seen = set()
for w in simplex(R, grid):
    key = tuple(np.round(w, 2))
    if key in seen:
        continue
    seen.add(key)
    m = blend_metric(w)
    if m < best['score']:
        best = {'name': 'grid', 'w': np.asarray(w, np.float32), 'score': m}
print(f'best grid blend   VAL {best["score"]:.5f}   w={np.round(best["w"],3)}  ({dict(zip(present, np.round(best["w"],3)))})')

# --- local coordinate-descent refinement ---
w = best['w'].copy()
step = 0.08
for _ in range(400):
    improved = False
    for i in range(R):
        for s in (step, -step):
            cand = w.copy(); cand[i] += s
            cand = np.clip(cand, 0, None)
            if cand.sum() == 0:
                continue
            cand /= cand.sum()
            m = blend_metric(cand)
            if m < best['score'] - 1e-7:
                best = {'name': 'refined', 'w': cand, 'score': m}; w = cand; improved = True
    if not improved:
        step *= 0.5
        if step < 1e-3:
            break
print(f'refined blend     VAL {best["score"]:.5f}   w={np.round(best["w"],3)}  ({dict(zip(present, np.round(best["w"],3)))})')

# --- per-variable (height vs AGB) weights ---
pv = np.tile(best['w'][:, None], (1, 2)).astype(np.float32)
pv_score = blend_metric(None, per_var=pv)
step = 0.08
for _ in range(400):
    improved = False
    for j in range(2):
        for i in range(R):
            for s in (step, -step):
                cand = pv.copy(); cand[i, j] += s
                cand[:, j] = np.clip(cand[:, j], 0, None)
                if cand[:, j].sum() == 0:
                    continue
                cand[:, j] /= cand[:, j].sum()
                m = blend_metric(None, per_var=cand)
                if m < pv_score - 1e-7:
                    pv, pv_score = cand, m; improved = True
    if not improved:
        step *= 0.5
        if step < 1e-3:
            break
print(f'per-variable blend VAL {pv_score:.5f}   height w={np.round(pv[:,0],3)}  agb w={np.round(pv[:,1],3)}')

USE_PER_VAR = pv_score < best['score'] - 1e-5
print('\n=> choosing', 'PER-VARIABLE blend' if USE_PER_VAR else 'single-weight blend')

## 5. Build the ensembled submission

In [ ]:
SEED_AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]

sample_sub = pd.read_csv(f'{COMP_ENS}/sample_submission.csv') if (COMP_ENS := '/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public') else None
print('sample rows:', len(sample_sub))

def blend_scenario(scn):
    stack = np.stack([preds[r][scn] for r in present], axis=0)   # (R,15,N,7)
    if USE_PER_VAR:
        out = stack.mean(0).copy()
        for j in range(2):
            out[..., j] = np.tensordot(pv[:, j], stack[..., j], axes=(0, 0))
        return out
    return np.tensordot(best['w'], stack, axes=(0, 0))

parts = []
for scn in ['ssp126', 'ssp585']:
    if not all(scn in preds[r] for r in present):
        print('  !! missing scenario', scn, 'in one of the runs — skipping'); continue
    p = np.maximum(blend_scenario(scn), 0.0)
    N = p.shape[1]
    ids = [f'{scn}_{site}_{age}' for age in SEED_AGES for site in range(N)]
    parts.append(pd.DataFrame({'id': ids,
                               'height': (p[:, :, 0] / Y_MEAN[0]).reshape(-1),
                               'agb':    (p[:, :, 1] / Y_MEAN[1]).reshape(-1)}))

pred_df = pd.concat(parts).set_index('id')
sub = sample_sub[['id']].join(pred_df, on='id')
assert not sub[['height', 'agb']].isna().any().any(), 'NaNs / missing ids — is a run/scenario missing?'
sub.to_csv(f'{OUT}/submission.csv', index=False)
print('wrote', f'{OUT}/submission.csv', '| rows', len(sub))
print(sub.head())
print('\nblend weights used:', dict(zip(present, np.round(best["w"], 3))) if not USE_PER_VAR
      else {'height': dict(zip(present, np.round(pv[:,0],3))), 'agb': dict(zip(present, np.round(pv[:,1],3)))})
print('expected VAL metric:', round(pv_score if USE_PER_VAR else best["score"], 5))